In [155]:

import os
from dotenv import find_dotenv, load_dotenv


_ = load_dotenv(find_dotenv(),override=True)


openai_key = os.getenv("OPENAI_API_KEY")
from openai import OpenAI

In [156]:
if openai_key:
    print(f"✅ Key loaded successfully! Length: {len(openai_key)}")
else:
    print("❌ Key is still None. Your .env file is either missing or misplaced.")


✅ Key loaded successfully! Length: 73


In [157]:
from langgraph.graph import StateGraph, END
from typing import TypedDict,Annotated
import operator
from langchain_core.messages import AnyMessage,SystemMessage, HumanMessage,ToolMessage,AIMessage
from langchain_openai import ChatOpenAI
#from langchain_openrouter import ChatOpenRouter
#from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.tavily_search import TavilySearchResults
#from langchain_tavily import TavilySearch

In [158]:
tool = TavilySearchResults(max_results=2)
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


In [159]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage],operator.add]

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

#memory = SqliteSaver.from_conn_string(":memory:")

In [161]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [165]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()

prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model = "openai/gpt-3.5-turbo",base_url="https://openrouter.ai/api/v1",api_key=openai_key)
#with SqliteSaver.from_conn_string(":memory:") as memory:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [166]:
thread = {"configurable": {"thread_id": "1"}}

In [167]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable":{"thread_id":"1"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v["messages"])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000108, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000108, 'upstream_inference_prompt_cost': 7.65e-05, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780628872-PjVqhWu1x8jdRrCJEPGh', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e95c0-5b92-7751-bc37-0e6c16e3284c-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_jsmBB798LxCW87Ty9uLqnS3z',

In [168]:
messages = [HumanMessage(content="What is the weather in LA?")]
thread = {"configurable":{"thread_id":"1"}}
for event in abot.graph.stream({"messages":messages},thread):
    for v in event.values():
        print(v["messages"])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 1316, 'total_tokens': 1337, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0006895, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0006895, 'upstream_inference_prompt_cost': 0.000658, 'upstream_inference_completions_cost': 3.15e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780628887-oNYqFhQJ8uz3caKxxs7l', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e95c0-98a8-7c62-ab3a-bd01fcdd1c21-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_bjBjRVm4YhBKbm3DCna4fcIi

In [171]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 2415, 'total_tokens': 2472, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.001293, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.001293, 'upstream_inference_prompt_cost': 0.0012075, 'upstream_inference_completions_cost': 8.55e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780635712-oeIv0w7tZyvn5JFuKsbo', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9628-bb57-7392-8f12-9c5db1f58e29-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_Hkry4Hk5Id

In [175]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
memory = InMemorySaver()
#memory = AsyncSqliteSaver.from_conn_string(":memory:")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [176]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

c:\Users\anura\AI\Projects\PythonAgent\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3746: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_ZDBdKQmAJ1LyhvHTccmFFK5a', 'type': 'tool_call'}
Back to the model!
The| weather| in| San| Francisco| shows| daily| high| temperatures| ranging| from| |61|°F| to| |81|°F| over| the| next| few| days|.| The| lowest| temperature| recorded| recently| was| |50|.|5|°F| and| the| highest| was| |80|.|1|°F|.| There| has| been| no| precipitation| reported| during| the| last| |15| days| in| San| Francisco|.|